# ICLR Analysis

This notebook grabs the representation_metrics.csv files from storage and plots metrics like
dirichlet energy over context length.


## Helper Functions

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import shutil
import os
import constants

### ICLR Line Plots

In [80]:

def advanced_line_plot(df: pd.DataFrame, x_col: str, y_col: str, y2_col: str = None, indicator_col: str = None, title: str = 'Advanced Line Plot', outpath: str = None, labels: list[str] = None, indicator_subset: list = None, window_multiplier=50):
    """
    Generates a line plot from a pandas DataFrame using Seaborn,
    with an optional secondary y-axis and an indicator variable.
    If an indicator is provided, it creates a separate subplot for each indicator value.
    The x-axis is plotted on a logarithmic scale.

    Args:
        df (pd.DataFrame): The input DataFrame.
        x_col (str): The column name for the x-axis. Must be numeric and positive for log scale.
        y_col (str): The column name for the primary y-axis.
        y2_col (str, optional): The column name for the secondary (twin) y-axis. Defaults to None.
        indicator_col (str, optional): The column name for the categorical indicator variable.
                                     If provided, creates a separate subplot for each category.
        title (str, optional): The title for the plot. Defaults to 'Advanced Line Plot'.
        labels: Optional labels to rename axis labels
        indicator_subset (list, optional): A list of specific values from the indicator_col to plot.
                                     If None, all unique values will be plotted. Defaults to None.
    """
    # --- Input Validation ---
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")
    
    required_cols = {x_col, y_col}
    df[x_col] = (df[x_col] * window_multiplier) + 50
    if y2_col:
        required_cols.add(y2_col)
    if indicator_col:
        required_cols.add(indicator_col)

    if not required_cols.issubset(df.columns):
        missing_cols = required_cols - set(df.columns)
        raise ValueError(f"The following columns are missing from the DataFrame: {missing_cols}")
    
    if not pd.api.types.is_numeric_dtype(df[x_col]) or (df[x_col] < 0).any():
        raise ValueError(f"x_col '{x_col}' must contain positive numeric data for a log scale.")

    # --- Plotting Setup ---
    sns.set_theme(style="darkgrid")

    if indicator_col:
        # --- Create a separate subplot for each indicator value ---
        all_indicator_values = sorted(df[indicator_col].unique())

        if indicator_subset:
            # Filter to only plot the subset of indicators provided
            indicator_values = [val for val in indicator_subset if val in all_indicator_values]
            if not indicator_values:
                 raise ValueError(f"None of the values in indicator_subset {indicator_subset} were found in the column '{indicator_col}'.")
        else:
            indicator_values = all_indicator_values
        
        n_indicators = len(indicator_values)
        
        fig, axes = plt.subplots(n_indicators, 1, figsize=(8, 6 * n_indicators), sharex=True, squeeze=False)
        axes = axes.flatten()
        if len(indicator_values) > 1:
          fig.suptitle(title, fontsize=16)
        else:
            axes[0].set_title(title, fontsize=18)

        for i, value in enumerate(indicator_values):
            ax1 = axes[i]
            ax2 = None  # Reset for each subplot
            group_df = df[df[indicator_col] == value]

            # Plot primary y-axis data
            handle_label = y_col if labels is None else labels[1]
            sns.lineplot(data=group_df, x=x_col, y=y_col, ax=ax1, label=handle_label, color='C0')
            
            # Only set title if there are multiple indicators
            if n_indicators > 1:
                if labels:
                    ax1.set_title(f'{labels[-1]} = {value}')
                    ax1.set_ylabel(labels[1], color='C0', fontsize=15)
                else:
                    ax1.set_title(f'{indicator_col} = {value}')
                    ax1.set_ylabel(y_col, color='C0', fontsize=15)
                ax1.tick_params(axis='y', colors='C0', size=14)
            else:
                if labels:
                    ax1.set_ylabel(labels[1], color='C0', fontsize=15)
                else:
                    ax1.set_ylabel(y_col, color='C0', fontsize=15)
                ax1.tick_params(axis='y', colors='C0', size=14)

            # Plot secondary y-axis data if provided
            if y2_col:
                ax2 = ax1.twinx()
                handle_label = y2_col if labels is None else labels[2]
                sns.lineplot(data=group_df, x=x_col, y=y2_col, ax=ax2, label=handle_label, color='C1', linestyle='--')
                if labels:
                    ax2.set_ylabel(labels[2], color='C1', fontsize=15)
                else:
                    ax2.set_ylabel(y2_col, color='C1', fontsize=15)
                ax2.tick_params(axis='y', colors='C1', size=14)
                ax2.grid(False)

            ax1.set_xscale('log')

            # Handle legends for each subplot
            handles1, labels1 = ax1.get_legend_handles_labels()
            if ax2:
                handles2, labels2 = ax2.get_legend_handles_labels()
                ax1.legend(handles1 + handles2, labels1 + labels2, loc='best')
                if ax2.get_legend(): ax2.get_legend().remove()
            else:
                ax1.legend(loc='best')
        
        if labels:
            axes[-1].set_xlabel(labels[0], fontsize=15)
        else:
            axes[-1].set_xlabel(x_col, fontsize=15) # Set common x-label on the last plot
        axes[-1].tick_params(axis='x', size=15)

    else:
        # --- Plot without an indicator variable (single plot) ---
        fig, ax1 = plt.subplots(figsize=(14, 7))
        ax1.set_title(title, fontsize=14)
        handle_label = y_col if labels is None else labels[1]
        sns.lineplot(data=df, x=x_col, y=y_col, ax=ax1, color='blue', label=handle_label)
        if labels:
            ax1.set_ylabel(labels[1], color='blue')
        else:
            ax1.set_ylabel(y_col, color='blue')
        ax1.tick_params(axis='y', colors='blue')
        
        if y2_col:
            ax2 = ax1.twinx()
            handle_label = y2_col if labels is None else labels[2]
            sns.lineplot(data=df, x=x_col, y=y2_col, ax=ax2, color='red', linestyle='--', label=handle_label)
            if labels:
                ax2.set_ylabel(labels[2], color='red')
            else:
                ax2.set_ylabel(y2_col, color='red')
            ax2.tick_params(axis='y', colors='red')
            ax2.grid(False)

        ax1.set_xscale('log')
        if labels:
            ax1.set_xlabel(labels[0], fontsize=14)
        else:
            ax1.set_xlabel(x_col, fontsize=14)

        # Combine legends
        handles1, labels1 = ax1.get_legend_handles_labels()
        try:
            handles2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(handles1 + handles2, labels1 + labels2, loc='best')
        except NameError:
            ax1.legend(loc='best')

    # --- Final Touches ---
    fig.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.savefig(f"./ICLR_{outpath}.pdf", format="pdf", bbox_inches="tight", dpi=600)
    if gfile.Exists(os.path.join(FIGURES_PATH, f"./ICLR_{outpath}.pdf")):
        gfile.Remove(os.path.join(FIGURES_PATH, f"./ICLR_{outpath}.pdf"))
    gfile.Copy(f"./ICLR_{outpath}.pdf", os.path.join(FIGURES_PATH, f"./ICLR_{outpath}.pdf"))
    plt.show()



In [32]:
import shutil

def graph_helper(path, y_col_1, y_col_2=None, indicator_col=None, title=None, outpath=None, labels=None, layer_subset=None):
  if os.path.exists("./results.csv"):
    os.remove("./results.csv")
  shutil.copy(path, "./results.csv")
  df = pd.read_csv("./results.csv")
  advanced_line_plot(df, "window_start", y_col_1, y_col_2, indicator_col, title, outpath=outpath, labels=labels, indicator_subset=layer_subset)

In [78]:
def plot_results(model, world_size, plot_both, dimension, prompt, act_window, norm_dirichlet, layer_subset, prepopulated=False):
  res_dir = "results"
  path = f"{constants.ROOT_DIR}/ICWM/{res_dir}/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/{prompt}/representation_metrics.csv"
  
  if dimension == 1:
    task_str = "Line Walk"
  else:
    task_str = "Grid Walk"
    
  if prepopulated:
    task_str += " Prefilled"

  if model == "gemma-3-27b-it":
    model_str = "Gemma-27b"
  elif model == "gemma-3-12b-it":
    model_str = "Gemma-12b"
  elif model == "gemma-3-4b-it":
    model_str = "Gemma-4b"
  elif model == "olmo-2-13b-it":
    model_str = "OLMo-13b"
  else:
    raise ValueError(f"Unsupported model: {model}")

  if len(layer_subset) > 1:
    title = f"{model_str} World Size {world_size} {task_str} Representations"
  else:
    title = f"{model_str} {task_str.replace(' Prefilled', '')} Representations (Layer {layer_subset[0]})"

  outpath = f"{model}_{world_size}_{dimension}_{act_window}_{prompt}_{'prepopulated' if prepopulated else 'not_prepopulated'}"
  if plot_both:
    if norm_dirichlet:
      graph_helper(path, "normalized_dirichlet_energy", "distance_correlation", "layer", title, outpath, ["Window Start", "Normalized Dirichlet Energy", "Distance Correlation", "Layer"], layer_subset=layer_subset)
    else:
      graph_helper(path, "dirichlet_energy", "distance_correlation", "layer", title, outpath, ["Window Start", "Dirichlet Energy", "Distance Correlation", "Layer"], layer_subset=layer_subset)
  else:
    if norm_dirichlet:
      graph_helper(path, "normalized_dirichlet_energy", indicator_col="layer", title=title, outpath=outpath, labels=["Window Start", "Normalized Dirichlet Energy", "Layer"], layer_subset=layer_subset)
    else:
      graph_helper(path, "dirichlet_energy", indicator_col="layer", title=title, outpath=outpath, labels=["Window Start", "Dirichlet Energy", "Layer"], layer_subset=layer_subset)


### Structure Difference Plots

In [16]:
import shutil

def plot_individual_structure_difference(model, world_size, dimension, act_window, layer_subset, metric="normalized_dirichlet_energy"):
  res_dir = "results"
  path_not_prepopulated = f"{constants.ROOT_DIR}/ICWM/{res_dir}/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/next_token_prediction/representation_metrics.csv"
  path_prepopulated = f"{constants.ROOT_DIR}/ICWM/{res_dir}/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/next_token_prediction_prepopulated/representation_metrics.csv"

  if os.path.exists("./results_not_prepopulated.csv"):
    os.remove("./results_not_prepopulated.csv")
  shutil.copy(path_not_prepopulated, "./results_not_prepopulated.csv")
  not_prepopulated_df = pd.read_csv("./results_not_prepopulated.csv")
  
  if os.path.exists("./results_prepopulated.csv"):
    os.remove("./results_prepopulated.csv")
  shutil.copy(path_prepopulated, "./results_prepopulated.csv")
  prepopulated_df = pd.read_csv("./results_prepopulated.csv")
  
  if dimension == 1:
    task_str = "Line Walk"
  else:
    task_str = "Grid Walk"

  title = f"{model} World Size {world_size} Dim {dimension} Structure Difference Plots"
  
  not_prepopulated_df["Prompt Type"] = ["Instruction"] * len(not_prepopulated_df)
  prepopulated_df["Prompt Type"] = ["Prepopulated"] * len(prepopulated_df)
  
  df = pd.concat([not_prepopulated_df, prepopulated_df])
  
  df = df[df["layer"].isin(layer_subset)]
  
  df["window_start"] = (df["window_start"] * 50) + 50
  
  sns.lineplot(data=df, x="window_start", y=metric, hue="Prompt Type", marker="o")
  if metric == "normalized_dirichlet_energy":
    plt.ylabel("Normalized Dirichlet Energy")
  else:
    plt.ylabel("Distance Correlation")
  plt.xlabel("Context Length")
  plt.title(title)
  plt.xscale("log")
  plt.tight_layout()
  plt.show()


In [60]:
import shutil

FIGURES_PATH = constants.ROOT_DIR + "/ICWM/Figures"

def plot_mean_structure_difference():
  
  windows = []
  correlations = []
  energies = []
  
  for model in ["gemma-3-27b-it", "gemma-3-12b-it", "gemma-3-4b-it", "olmo-2-13b-it"]:
    for world_size in [16, 25]:
      for dimension in [1, 2]:
        act_window = 150 if dimension == 1 else 50
        if dimension == 1:
          prompt_str = "next_token_prediction_1d"
        else:
          prompt_str = "next_token_prediction"
        path_not_prepopulated = f"{constants.ROOT_DIR}/ICWM/results/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/{prompt_str}/representation_metrics.csv"
        path_prepopulated = f"{constants.ROOT_DIR}/ICWM/results/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/{prompt_str}_prepopulated/representation_metrics.csv"
  
        if os.path.exists("./results_not_prepopulated.csv"):
          os.remove("./results_not_prepopulated.csv")
        shutil.copy(path_not_prepopulated, "./results_not_prepopulated.csv")
        not_prepopulated_df = pd.read_csv("./results_not_prepopulated.csv")
        
        if os.path.exists("./results_prepopulated.csv"):
          os.remove("./results_prepopulated.csv")
        shutil.copy(path_prepopulated, "./results_prepopulated.csv")
        prepopulated_df = pd.read_csv("./results_prepopulated.csv")
        
        layers = not_prepopulated_df["layer"].unique().tolist()
        layers.sort()
        layer_to_plot = layers[-2]
        
        not_prepopulated_df = not_prepopulated_df[not_prepopulated_df["layer"] == layer_to_plot]
        prepopulated_df = prepopulated_df[prepopulated_df["layer"] == layer_to_plot]
        
        not_prepopulated_df["window"] = (not_prepopulated_df["window_start"] * 50) + 50
        prepopulated_df["window"] = (prepopulated_df["window_start"] * 50) + 50
        
        for window in not_prepopulated_df["window"].unique().tolist():
          windows.append(window)
          
          not_prepopulated_corr = np.mean(not_prepopulated_df[not_prepopulated_df["window"] == window]["distance_correlation"])
          prepopulated_corr = np.mean(prepopulated_df[prepopulated_df["window"] == window]["distance_correlation"])
          correlations.append(not_prepopulated_corr / prepopulated_corr)
          
          not_prepopulated_energy = np.mean(not_prepopulated_df[not_prepopulated_df["window"] == window]["normalized_dirichlet_energy"])
          prepopulated_energy = np.mean(prepopulated_df[prepopulated_df["window"] == window]["normalized_dirichlet_energy"])
          energies.append(not_prepopulated_energy / prepopulated_energy)
  
  
  df = pd.DataFrame({"window": windows, "correlation": correlations, "energy": energies})

  sns.set_theme(style="darkgrid")

  # Initialize the figure with 2 subplots sharing the x-axis
  fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

  # Plot Column A on the first subplot
  sns.lineplot(ax=axes[0], x=df['window'], y=df['correlation'], color='tab:blue', linewidth=2.5)
  axes[0].set_ylabel('Distance Correlation Ratio')
  axes[0].set_title('Ratio of ICLR Metrics (Instruction / Prefilled)', fontsize=16)

  # Plot Column B on the second subplot
  sns.lineplot(ax=axes[1], x=df['window'], y=df['energy'], color='tab:orange', linewidth=2.5)
  axes[1].set_ylabel('Normalized Dirichlet Energy Ratio')
  axes[1].set_xlabel('Context Length')

  # Adjust layout to prevent overlap
  plt.tight_layout()
  plt.xscale("log")

  plt.savefig("./ICLR_Structure_Difference.jpg", format="jpg", bbox_inches="tight")
  if os.path.exists(os.path.join(FIGURES_PATH, "ICLR_Structure_Difference.jpg")):
    os.remove(os.path.join(FIGURES_PATH, "ICLR_Structure_Difference.jpg"))
  shutil.copy("./ICLR_Structure_Difference.jpg", os.path.join(FIGURES_PATH, "ICLR_Structure_Difference.jpg"))


In [17]:
plot_individual_structure_difference("gemma-3-27b-it", 25, 2, 50, layer_subset=[60, 62])

In [18]:
plot_individual_structure_difference("gemma-3-27b-it", 25, 2, 50, layer_subset=[60, 62], metric="distance_correlation")

In [27]:
plot_mean_structure_difference()


### Context Length Finding Functions

In [50]:
def get_max_correlation(model, world_size, dimension, prompt, act_window, layers):
  res_dir = "results"
  path = f"{constants.ROOT_DIR}/ICWM/{res_dir}/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/{prompt}/representation_metrics.csv"
  df = pd.read_csv(path)
  df = df[df["layer"].isin(layers)]
  grps = df.groupby("window_start")
  distance_correlations = []
  for window_start, grp in grps:
    distance_correlations.append(np.mean(grp["distance_correlation"]))

  # This is correct, as the context window refers to the number of adjacencies
  # in the structure presentation. The activation window is either 50 or 150 
  # because 2D structures are presented as random walks of tokens, so token 
  # count and adjacency count are equivalent. 1D structures are presented as
  # random edges + a newline, so each adjacency is 3 tokens, hence 150 activation
  # window.
  print(f"{model} World Size {world_size} Dim {dimension} prompt {prompt} Best Context length: {(np.argmax(distance_correlations) + 1) * 50}")    


In [51]:
def get_min_energy(model, world_size, dimension, prompt, act_window, layers):
  res_dir = "results"
  path = f"{constants.ROOT_DIR}/ICWM/{res_dir}/{model}/world_size_{world_size}/dimension_{dimension}/activation_window_{act_window}/{prompt}/representation_metrics.csv"
  df = pd.read_csv(path)
  df = df[df["layer"].isin(layers)]
  grps = df.groupby("window_start")
  energy = []
  for window_start, grp in grps:
    assert window_start == len(energy)
    energy.append(np.mean(grp["normalized_dirichlet_energy"]))
  
  # This is correct, as the context window refers to the number of adjacencies
  # in the structure presentation. The activation window is either 50 or 150 
  # because 2D structures are presented as random walks of tokens, so token 
  # count and adjacency count are equivalent. 1D structures are presented as
  # random edges + a newline, so each adjacency is 3 tokens, hence 150 activation
  # window.
  print(f"{model} World Size {world_size} Dim {dimension} prompt {prompt} Best Context length: {(np.argmin(energy) + 1) * 50}")    


### @TODO Get Max Correlation after running alternative NTP ICLR Analysis

In [ ]:
get_max_correlation("gemma-3-4b-it", 16, 1, "alternative_next_token_prediction_1d", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 1, "alternative_next_token_prediction_1d", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 16, 2, "alternative_next_token_prediction", 50, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 2, "alternative_next_token_prediction", 50, layers=[32, 34])

In [ ]:
get_max_correlation("gemma-3-12b-it", 16, 1, "alternative_next_token_prediction_1d", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 1, "alternative_next_token_prediction_1d", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 16, 2, "alternative_next_token_prediction", 50, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 2, "alternative_next_token_prediction", 50, layers=[46, 48])

In [ ]:
get_max_correlation("gemma-3-27b-it", 16, 1, "alternative_next_token_prediction_1d", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 1, "alternative_next_token_prediction_1d", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 16, 2, "alternative_next_token_prediction", 50, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 2, "alternative_next_token_prediction", 50, layers=[60, 62])


In [ ]:
get_max_correlation("olmo-2-13b-it", 16, 1, "alternative_next_token_prediction_1d", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 1, "alternative_next_token_prediction_1d", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 16, 2, "alternative_next_token_prediction", 50, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 2, "alternative_next_token_prediction", 50, layers=[40, 38])

### Get Context Length that Results in Max Correlations

In [52]:
get_max_correlation("gemma-3-4b-it", 16, 1, "next_token_prediction_1d", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 1, "next_token_prediction_1d", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 16, 2, "next_token_prediction", 50, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 2, "next_token_prediction", 50, layers=[32, 34])

In [1]:
get_max_correlation("gemma-3-4b-it", 16, 1, "next_token_prediction_1d_prepopulated", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 1, "next_token_prediction_1d_prepopulated", 150, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 16, 2, "next_token_prediction_prepopulated", 50, layers=[32, 34])
get_max_correlation("gemma-3-4b-it", 25, 2, "next_token_prediction_prepopulated", 50, layers=[32, 34])

In [54]:
get_max_correlation("gemma-3-12b-it", 16, 1, "next_token_prediction_1d", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 1, "next_token_prediction_1d", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 16, 2, "next_token_prediction", 50, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 2, "next_token_prediction", 50, layers=[46, 48])

In [55]:
get_max_correlation("gemma-3-12b-it", 16, 1, "next_token_prediction_1d_prepopulated", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 1, "next_token_prediction_1d_prepopulated", 150, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 16, 2, "next_token_prediction_prepopulated", 50, layers=[46, 48])
get_max_correlation("gemma-3-12b-it", 25, 2, "next_token_prediction_prepopulated", 50, layers=[46, 48])

In [56]:
get_max_correlation("gemma-3-27b-it", 16, 1, "next_token_prediction_1d", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 1, "next_token_prediction_1d", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 16, 2, "next_token_prediction", 50, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 2, "next_token_prediction", 50, layers=[60, 62])


In [57]:
get_max_correlation("gemma-3-27b-it", 16, 1, "next_token_prediction_1d_prepopulated", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 1, "next_token_prediction_1d_prepopulated", 150, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 16, 2, "next_token_prediction_prepopulated", 50, layers=[60, 62])
get_max_correlation("gemma-3-27b-it", 25, 2, "next_token_prediction_prepopulated", 50, layers=[60, 62])


In [58]:
get_max_correlation("olmo-2-13b-it", 16, 1, "next_token_prediction_1d", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 1, "next_token_prediction_1d", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 16, 2, "next_token_prediction", 50, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 2, "next_token_prediction", 50, layers=[40, 38])

In [59]:
get_max_correlation("olmo-2-13b-it", 16, 1, "next_token_prediction_1d_prepopulated", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 1, "next_token_prediction_1d_prepopulated", 150, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 16, 2, "next_token_prediction_prepopulated", 50, layers=[40, 38])
get_max_correlation("olmo-2-13b-it", 25, 2, "next_token_prediction_prepopulated", 50, layers=[40, 38])

## Plot Results

In [57]:
plot_results("gemma-3-4b-it", 16, True, 1, "next_token_prediction_1d", 150, True, [32, 34])
plot_results("gemma-3-4b-it", 16, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [32, 34], prepopulated=True)
plot_results("gemma-3-4b-it", 25, True, 1, "next_token_prediction_1d", 150, True, [32, 34])
plot_results("gemma-3-4b-it", 25, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [32, 34], prepopulated=True)
plot_results("gemma-3-4b-it", 16, True, 2, "next_token_prediction", 50, True, [32, 34])
plot_results("gemma-3-4b-it", 16, True, 2, "next_token_prediction_prepopulated", 50, True, [32, 34], prepopulated=True)
plot_results("gemma-3-4b-it", 25, True, 2, "next_token_prediction", 50, True, [32, 34])
plot_results("gemma-3-4b-it", 25, True, 2, "next_token_prediction_prepopulated", 50, True, [32, 34], prepopulated=True)

In [56]:
plot_results("gemma-3-12b-it", 16, True, 1, "next_token_prediction_1d", 150, True, [46, 48])
plot_results("gemma-3-12b-it", 16, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [46, 48], prepopulated=True)
plot_results("gemma-3-12b-it", 25, True, 1, "next_token_prediction_1d", 150, True, [46, 48])
plot_results("gemma-3-12b-it", 25, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [46, 48], prepopulated=True)
plot_results("gemma-3-12b-it", 16, True, 2, "next_token_prediction", 50, True, [46, 48])
plot_results("gemma-3-12b-it", 16, True, 2, "next_token_prediction_prepopulated", 50, True, [46, 48], prepopulated=True)
plot_results("gemma-3-12b-it", 25, True, 2, "next_token_prediction", 50, True, [46, 48])
plot_results("gemma-3-12b-it", 25, True, 2, "next_token_prediction_prepopulated", 50, True, [46], prepopulated=True)

In [81]:
plot_results("gemma-3-12b-it", 25, True, 2, "next_token_prediction_prepopulated", 50, True, [46], prepopulated=True)

In [55]:
plot_results("gemma-3-27b-it", 16, True, 1, "next_token_prediction_1d", 150, True, [60, 62])
plot_results("gemma-3-27b-it", 16, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [60, 62], prepopulated=True)
plot_results("gemma-3-27b-it", 25, True, 1, "next_token_prediction_1d", 150, True, [60, 62])
plot_results("gemma-3-27b-it", 25, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [60, 62], prepopulated=True)
plot_results("gemma-3-27b-it", 16, True, 2, "next_token_prediction", 50, True, [60, 62])
plot_results("gemma-3-27b-it", 16, True, 2, "next_token_prediction_prepopulated", 50, True, [60, 62], prepopulated=True)
plot_results("gemma-3-27b-it", 25, True, 2, "next_token_prediction", 50, True, [60, 62])
plot_results("gemma-3-27b-it", 25, True, 2, "next_token_prediction_prepopulated", 50, True, [60, 62], prepopulated=True)

In [58]:
plot_results("olmo-2-13b-it", 16, True, 1, "next_token_prediction_1d", 150, True, [40, 38])
plot_results("olmo-2-13b-it", 16, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [40, 38], prepopulated=True)
plot_results("olmo-2-13b-it", 25, True, 1, "next_token_prediction_1d", 150, True, [40, 38])
plot_results("olmo-2-13b-it", 25, True, 1, "next_token_prediction_1d_prepopulated", 150, True, [40, 38], prepopulated=True)
plot_results("olmo-2-13b-it", 16, True, 2, "next_token_prediction", 50, True, [40, 38])
plot_results("olmo-2-13b-it", 16, True, 2, "next_token_prediction_prepopulated", 50, True, [40, 38], prepopulated=True)
plot_results("olmo-2-13b-it", 25, True, 2, "next_token_prediction", 50, True, [40, 38])
plot_results("olmo-2-13b-it", 25, True, 2, "next_token_prediction_prepopulated", 50, True, [40, 38], prepopulated=True)